# DermSAM Training Notebook
**Before running:** Runtime → Change runtime type → T4 GPU

## 1. Mount Google Drive

In [ ]:
# ============================================================
# FULL SESSION RESET — run this cell every time Colab resets
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os

# Pull latest code
if os.path.exists('/content/dermSAM'):
    !cd /content/dermSAM && git pull
else:
    !git clone https://github.com/theomalaper/dermSAM.git /content/dermSAM

# Install deps
!pip install -q -r /content/dermSAM/requirements.txt
!pip install -q git+https://github.com/facebookresearch/segment-anything.git

# Unzip data if not already done
os.makedirs('/content/data', exist_ok=True)
if not os.path.exists('/content/data/ISIC2018_Task1-2_Training_Input'):
    print('Unzipping images...')
    !unzip -q /content/drive/MyDrive/dermSAM/ISIC2018_Task1-2_Training_Input.zip -d /content/data/
if not os.path.exists('/content/data/ISIC2018_Task1_Training_GroundTruth'):
    print('Unzipping masks...')
    !unzip -q /content/drive/MyDrive/dermSAM/ISIC2018_Task1_Training_GroundTruth.zip -d /content/data/

# Symlink data into repo
os.makedirs('/content/dermSAM/data', exist_ok=True)
!ln -sf /content/data/ISIC2018_Task1-2_Training_Input /content/dermSAM/data/ISIC2018_Task1-2_Training_Input
!ln -sf /content/data/ISIC2018_Task1_Training_GroundTruth /content/dermSAM/data/ISIC2018_Task1_Training_GroundTruth
!ln -sf /content/drive/MyDrive/dermSAM/checkpoints /content/dermSAM/checkpoints

# Verify
import cv2, pandas as pd
df = pd.read_csv('/content/dermSAM/data/splits/train.csv')
img = cv2.imread(df.iloc[0]['image'])
print(f'Images: {"OK" if img is not None else "FAILED"} {img.shape if img is not None else ""}')
print(f'Checkpoints: {os.listdir("/content/dermSAM/checkpoints")}')
print('Setup complete.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone repo

In [ ]:
!git clone https://github.com/YOUR_USERNAME/dermSAM.git
%cd dermSAM

## 3. Install dependencies

In [ ]:
!pip install -q -r requirements.txt
!pip install -q git+https://github.com/facebookresearch/segment-anything.git
print('Done')

## 4. Upload data zip from your Mac
Run this cell, click 'Choose Files', select `ISIC2018_Task1-2_Training_Input.zip`. Will take ~20-30 min.

In [ ]:
from google.colab import files
print('Select ISIC2018_Task1-2_Training_Input.zip from your Mac')
uploaded = files.upload()

## 5. Unzip data

In [ ]:
!unzip -q ISIC2018_Task1-2_Training_Input.zip -d data/
!echo "Images: $(ls data/ISIC2018_Task1-2_Training_Input/*.jpg | wc -l)"

## 6. Back up zips to Drive (skip if already there)

In [ ]:
# Zip already lives on Drive — nothing to back up
print('Skipped — zip sourced directly from Drive')

## 7. Link checkpoints from Drive

In [ ]:
drive_data = '/content/drive/MyDrive/dermSAM'

# Link checkpoints
!ln -sf "{drive_data}/checkpoints" checkpoints

# Unzip ground truth masks if not already done
import os
if not os.path.exists('data/ISIC2018_Task1_Training_GroundTruth'):
    !unzip -q "{drive_data}/ISIC2018_Task1_Training_GroundTruth.zip" -d data/
    print('Masks unzipped')
else:
    print('Masks already unzipped')

!ls checkpoints/
!echo "Masks: $(ls data/ISIC2018_Task1_Training_GroundTruth/*.png | wc -l)"

## 8. Verify GPU + run tests

In [ ]:
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
!python3 -m pytest tests/ -v --tb=short

## 9. wandb login

In [ ]:
import wandb
wandb.login(key='YOUR_WANDB_API_KEY_HERE')

## 10. Train UNet baseline

In [ ]:
!python3 src/train.py \
    --model unet \
    --epochs 30 \
    --lr 1e-4 \
    --batch-size 16 \
    --scheduler plateau \
    --amp

## 11. Train Localizer

In [ ]:
!python3 src/train.py \
    --model localizer \
    --epochs 20 \
    --lr 1e-4 \
    --batch-size 32 \
    --scheduler plateau \
    --amp

## 12. Fine-tune MedSAM decoder

In [ ]:
!python3 src/train.py \
    --model medsam \
    --epochs 20 \
    --lr 1e-4 \
    --batch-size 4 \
    --scheduler cosine \
    --amp \
    --grad-accum 4 \
    --clip-grad 1.0 \
    --freeze-encoder

## 13. Run full benchmark

In [ ]:
!python3 src/evaluate.py --all --output outputs/metrics/benchmark.csv

## 14. Push results to GitHub

In [ ]:
!git config user.email 'your@email.com'
!git config user.name 'Your Name'
!git add outputs/ data/splits/ docs/results_log.md checkpoints/best_unet.pth checkpoints/best_localizer.pth
!git commit -m 'Add training results and benchmark outputs'
!git push